In [1]:
import sympy

import jax
import jax.numpy as jnp

In [2]:


_lookup = {
    jax.lax.mul_p: sympy.Mul,
    jax.lax.add_p: sympy.MatAdd,  # Note: jax.lax.add was originally mapped to both sympy.Add and sympy.MatAdd, keeping one
    jax.lax.div_p: sympy.div,
    jax.lax.abs_p: sympy.Abs,
    jax.lax.sign_p: sympy.sign,
    jax.lax.ceil_p: sympy.ceiling,
    jax.lax.floor_p: sympy.floor,
    jax.lax.log_p: sympy.log,
    jax.lax.exp_p: sympy.exp,
    jax.lax.sqrt_p: sympy.sqrt,
    jax.lax.cos_p: sympy.cos,
    jax.lax.acos_p: sympy.acos,
    jax.lax.sin_p: sympy.sin,
    jax.lax.asin_p: sympy.asin,
    jax.lax.tan_p: sympy.tan,
    jax.lax.atan_p: sympy.atan,
    jax.lax.atan2_p: sympy.atan2,
    jax.lax.cosh_p: sympy.cosh,
    jax.lax.acosh_p: sympy.acosh,
    jax.lax.sinh_p: sympy.sinh,
    jax.lax.asinh_p: sympy.asinh,
    jax.lax.tanh_p: sympy.tanh,
    jax.lax.atanh_p: sympy.atanh,
    jax.lax.pow_p: sympy.Pow,
    jax.lax.integer_pow_p: sympy.Pow,
    jax.lax.real_p: sympy.re,
    jax.lax.imag_p: sympy.im,
    jax.lax.erf_p: sympy.erf,
    jax.lax.eq_p: sympy.Eq,
    jax.lax.ne_p: sympy.Ne,
    jax.lax.gt_p: sympy.StrictGreaterThan,
    jax.lax.lt_p: sympy.StrictLessThan,
    jax.lax.le_p: sympy.LessThan,
    jax.lax.ge_p: sympy.GreaterThan,
    jax.lax.max_p: sympy.Max,
    jax.lax.min_p: sympy.Min,
    jax.lax.add_p: sympy.MatAdd,  # Duplicate, noted for completeness
}

_constant_lookup = {
    jnp.e: sympy.E,
    jnp.pi: sympy.pi,
    jnp.euler_gamma: sympy.EulerGamma,
    1j: sympy.I,
}

In [3]:
from probjax.core.transformation import symbolify, lambdaify

In [14]:


def f(x):
    return jnp.exp(x) + jnp.sin(x)


f_sym = symbolify(f)

In [15]:
expr = f_sym(jnp.ones((10,)))

[x]
{}
[]
[x]
{}
[]
[exp(x), sin(x)]
{}
[]


In [16]:
expr

exp(x) + sin(x)

In [40]:
sympy.symarray('x', (20,))

array([x_0, x_1, x_2, x_3, x_4, x_5, x_6, x_7, x_8, x_9, x_10, x_11, x_12,
       x_13, x_14, x_15, x_16, x_17, x_18, x_19], dtype=object)

In [13]:
sympy.integrate(f_sym(1.0), *expr.free_symbols, 0, 1)

[x]
{}
[]
[x]
{}
[]
[exp(x), sin(x)]
{}
[]


ValueError: Invalid limits given: (x, 0, 1)

In [88]:

x = sympy.Matrix(sympy.symbols('x1:4'))
A = sympy.MatrixSymbol("A", 3,3)

A@(x@x.T)@A.T

A*Matrix([
[x1**2, x1*x2, x1*x3],
[x1*x2, x2**2, x2*x3],
[x1*x3, x2*x3, x3**2]])*A.T

In [113]:
x = sympy.Symbol('x', real=True)
y = sympy.Symbol('y', real=True)
M = sympy.Matrix([[x, y], [1, 0]])
M_square = M@M
M.integrate((x,0,2 ))


Matrix([
[2, 2*y],
[2,   0]])

In [114]:
M_square

Matrix([
[x**2 + y, x*y],
[       x,   y]])

In [105]:
def f(x):
    y = x**2 + 2
    return (y + x)**3

jaxpr = jax.make_jaxpr(f)(jnp.array(2.0))

In [35]:
jax.lax.pow

<function jax._src.lax.lax.pow(x: 'ArrayLike', y: 'ArrayLike') -> 'Array'>

In [36]:
jax.lax.integer_pow

<function jax._src.lax.lax.integer_pow(x: 'ArrayLike', y: 'int') -> 'Array'>

In [37]:
invars = jaxpr.jaxpr.invars
consts = jaxpr.jaxpr.constvars

In [38]:
eq.params

{}

In [39]:
syms = {invar: sympy.Symbol(f"x_{i}") for i, invar in enumerate(invars)}

for eq in jaxpr.eqns:
    sympy_eq = _lookup[eq.primitive] 
    params = list(eq.params.values())
    sym_invars = [syms[invar] if not isinstance(invar, jax.core.Literal) else invar.val for invar in eq.invars]
    sym_eq = sympy_eq(*sym_invars,*params)
    syms[eq.outvars[0]] = sym_eq

In [40]:
syms

{a: x_0,
 b: x_0**2,
 c: x_0**2 + 2.0,
 d: x_0**2 + x_0 + 2.0,
 e: 8.0*(0.5*x_0**2 + 0.5*x_0 + 1)**3}

In [95]:
?ImageFolder

Init signature:
ImageFolder(
    root: str,
    transform: Optional[Callable] = None,
    target_transform: Optional[Callable] = None,
    loader: Callable[[str], Any] = <function default_loader at 0x7f91f43979a0>,
    is_valid_file: Optional[Callable[[str], bool]] = None,
)
Docstring:     
A generic data loader where the images are arranged in this way by default: ::

    root/dog/xxx.png
    root/dog/xxy.png
    root/dog/[...]/xxz.png

    root/cat/123.png
    root/cat/nsdf3.png
    root/cat/[...]/asd932_.png

This class inherits from :class:`~torchvision.datasets.DatasetFolder` so
the same methods can be overridden to customize the dataset.

Args:
    root (string): Root directory path.
    transform (callable, optional): A function/transform that  takes in an PIL image
        and returns a transformed version. E.g, ``transforms.RandomCrop``
    target_transform (callable, optional): A function/transform that takes in the
        target and transforms it.
    loader (callable, opti

In [ ]:
/mnt/qb/macke/mgloeckler90/imagenet_cs3step

In [48]:
eq.params

{'y': 2}

In [117]:
from sympy import symbols, integrate, lambdify

# Define the variable
x = symbols('x')
a = symbols('a')
b = symbols('b')

# Define the function
f = x**2



# Integrate the function from 0 to 1
integral = integrate(f, (x, a, b))

integral

-a**3/3 + b**3/3

In [131]:
jax.jit(lambdify((a,b), integral, modules="jax"))(0,2)

Array(2.6666667, dtype=float32, weak_type=True)

In [16]:
integral

-a**3/3 + b**3/3

In [28]:
_f = lambdify((a,b), integral, modules='jax')

In [30]:
jax.make_jaxpr(_f)(1.,2.)

{ lambda ; a:f32[] b:f32[]. let
    c:f32[] = integer_pow[y=3] a
    d:f32[] = mul -0.3333333333333333 c
    e:f32[] = integer_pow[y=3] b
    f:f32[] = mul 0.3333333333333333 e
    g:f32[] = add d f
  in (g,) }

In [40]:
import sympy as sp
from collections import defaultdict, deque

def build_dependency_graph(expr):
    # Function to build a dependency graph from the expression
    graph = defaultdict(set)
    visited = set()
    
    def add_edges(node):
        if node in visited:
            return
        visited.add(node)
        for arg in node.args:
            graph[node].add(arg)
            add_edges(arg)
    
    add_edges(expr)
    return graph

def topological_sort(graph):
    # Function to perform topological sort on the graph
    in_degree = defaultdict(int)
    
    # Initialize in_degree for all nodes
    for u in graph:
        for v in graph[u]:
            in_degree[v] += 1
        if u not in in_degree:
            in_degree[u] = 0
    
    queue = deque([u for u in in_degree if in_degree[u] == 0])
    sorted_list = []
    
    while queue:
        u = queue.popleft()
        sorted_list.append(u)
        for v in graph[u]:
            in_degree[v] -= 1
            if in_degree[v] == 0:
                queue.append(v)
                
    return reversed(sorted_list)

# Example usage
x, y, z = sp.symbols('x y z')
expr = (x + y) * sp.sin(z) + sp.exp(x * y)

# Step 1: Create the dependency graph
graph = build_dependency_graph(expr)

# Step 2: Get the topologically sorted list of operations
sorted_operations = topological_sort(graph)

# Print the sorted operations
for op in sorted_operations:
    print(op)


x
y
z
x + y
sin(z)
x*y
(x + y)*sin(z)
exp(x*y)
(x + y)*sin(z) + exp(x*y)
